# 00 — Unified Mapping, CWE Head & LoRA Model

**Run A–Z. No prior state required beyond `FINAL_train.jsonl` and `FINAL_val.jsonl`.**

### What this notebook does
1. **Explore data** — inspect raw records, understand schema and distributions
2. **Validate** — check structure integrity of both splits
3. **Normalize** — add metadata fields, merge into `UNIFIED.jsonl` + mappings + metadata
4. **CWE head** — define 7-class classification head
5. **Model** — LoRA-wrapped GraphCodeBERT with all heads
6. **Smoke test** — verify forward pass, shapes, loss

### Platform support
| Platform | Backend | Notes |
|---|---|---|
| Apple Silicon (M1/M2/M3) | **MLX** | Native ARM — fast inference & training |
| Mac Intel / Linux CPU | **PyTorch CPU** | Slow but functional |
| Google Colab / CUDA GPU | **PyTorch CUDA** | Recommended for full training |

> **Apple Silicon note**: MLX cells are marked `[MLX — Apple Silicon only]`. They use `mlx` and `mlx_lm` which run natively on the Metal GPU. On other platforms, the notebook falls back to PyTorch automatically.

## 0 — Install Dependencies

In [1]:
import subprocess, sys, platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in __import__("os").environ

print(f"Platform : {platform.system()} {platform.machine()}")
print(f"Apple Silicon : {IS_APPLE_SILICON}")
print(f"Colab : {IS_COLAB}")

# Core deps — always needed
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.40",
        "peft>=0.10",
        "torch",
        "datasets",
    ],
    check=False,
)

# MLX — only on Apple Silicon
if IS_APPLE_SILICON:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mlx>=0.16", "mlx-lm>=0.16"],
        check=False,
    )
    print("MLX installed for Apple Silicon")

print("Done.")

Platform : Darwin arm64
Apple Silicon : True
Colab : False
MLX installed for Apple Silicon
Done.


## 1 — Paths & Helpers

In [2]:
import os, json, platform, sys
from pathlib import Path
from collections import Counter
from typing import List, Dict, Any

# ── Detect environment ──────────────────────────────────────────────────────
IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"

try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("Colab — Drive mounted")
except ImportError:
    BASE_DIR = Path(os.path.dirname(os.path.abspath("__file__")))
    # fallback: cwd
    if not (BASE_DIR / "datasets").exists():
        BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f"Local — BASE_DIR: {BASE_DIR}")

DATASETS_DIR = BASE_DIR / "datasets"
assert DATASETS_DIR.exists(), f"datasets/ not found at {DATASETS_DIR}"

TRAIN_PATH = DATASETS_DIR / "FINAL_train.jsonl"
VAL_PATH = DATASETS_DIR / "FINAL_val.jsonl"
assert TRAIN_PATH.exists(), f"Missing: {TRAIN_PATH}"
assert VAL_PATH.exists(), f"Missing: {VAL_PATH}"


# ── I/O helpers ─────────────────────────────────────────────────────────────
def read_jsonl(path) -> List[Dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, records: List[Dict]):
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


print(f"Apple Silicon : {IS_APPLE_SILICON}")
print(f"Colab         : {IS_COLAB}")
print(f"TRAIN_PATH    : {TRAIN_PATH}")
print(f"VAL_PATH      : {VAL_PATH}")

Local — BASE_DIR: /Users/anas/Projects/code-security-identifier
Apple Silicon : True
Colab         : False
TRAIN_PATH    : /Users/anas/Projects/code-security-identifier/datasets/FINAL_train.jsonl
VAL_PATH      : /Users/anas/Projects/code-security-identifier/datasets/FINAL_val.jsonl


## 2 — Explore Raw Data (first look)

In [3]:
import statistics

train_raw = read_jsonl(TRAIN_PATH)
val_raw = read_jsonl(VAL_PATH)
all_raw = train_raw + val_raw

# ── Schema ───────────────────────────────────────────────────────────────────
sample = all_raw[0]
print("=== RAW RECORD FIELDS ===")
for k, v in sample.items():
    if isinstance(v, list):
        print(f"  {k:<18} list[{type(v[0]).__name__}]  len={len(v)}")
    else:
        print(f"  {k:<18} {type(v).__name__}  = {v!r}")

print()
print("=== FIRST RECORD (truncated) ===")
print(f'  source  : {sample["dataset_source"]}')
print(f'  cwe_id  : {sample["cwe_id"]}')
print(f'  #lines  : {len(sample["lines"])}')
print(f'  labels  : {sample["label"]}')
print(f'  line[0] : {sample["lines"][0]!r}')

# ── Counts ───────────────────────────────────────────────────────────────────
print()
print("=== RECORD COUNTS ===")
print(f"  train : {len(train_raw):,}")
print(f"  val   : {len(val_raw):,}")
print(f"  total : {len(all_raw):,}")

# ── CWE distribution ─────────────────────────────────────────────────────────
cwes = Counter(r["cwe_id"] for r in all_raw)
print()
print("=== CWE DISTRIBUTION ===")
for cwe, cnt in cwes.most_common():
    bar = "█" * (cnt // 50)
    print(f"  {cwe:<12} {cnt:>5,}  {bar}")

# ── Sources ───────────────────────────────────────────────────────────────────
srcs = Counter(r["dataset_source"] for r in all_raw)
print()
print("=== DATASET SOURCES ===")
for src, cnt in srcs.most_common():
    print(f"  {src:<16} {cnt:>5,}")

# ── Statement stats ───────────────────────────────────────────────────────────
total_stmts = sum(len(r["label"]) for r in all_raw)
vuln_stmts = sum(sum(r["label"]) for r in all_raw)
vuln_funcs = sum(1 for r in all_raw if any(r["label"]))
lengths = [len(r["label"]) for r in all_raw]

print()
print("=== STATEMENT STATS ===")
print(f"  total statements : {total_stmts:,}")
print(f"  vuln statements  : {vuln_stmts:,}  ({100*vuln_stmts/total_stmts:.1f}%)")
print(f"  vuln functions   : {vuln_funcs:,}  ({100*vuln_funcs/len(all_raw):.1f}%)")
print(
    f"  stmts/func — min:{min(lengths)}  max:{max(lengths)}  mean:{statistics.mean(lengths):.1f}  median:{statistics.median(lengths)}"
)

# ── Label sanity ──────────────────────────────────────────────────────────────
bad = [(i, v) for i, r in enumerate(all_raw) for v in r["label"] if v not in (0, 1)]
print()
print(f"  invalid label values : {len(bad)}  (expect 0)")

=== RAW RECORD FIELDS ===
  lines              list[str]  len=17
  raw_lines          list[str]  len=17
  label              list[int]  len=17
  type               list[str]  len=17
  cwe_id             str  = 'unknown'
  dataset_source     str  = 'funclevel'

=== FIRST RECORD (truncated) ===
  source  : funclevel
  cwe_id  : unknown
  #lines  : 17
  labels  : [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  line[0] : 'def _download_artifact(artifact_path):'

=== RECORD COUNTS ===
  train : 3,677
  val   : 408
  total : 4,085

=== CWE DISTRIBUTION ===
  unknown      2,036  ████████████████████████████████████████
  CWE-089        589  ███████████
  CWE-079        404  ████████
  CWE-022        232  ████
  CWE-601        209  ████
  CWE-077        194  ███
  CWE-352        181  ███
  CWE-094        122  ██
  CWE-611          6  
  CWE-020          6  
  CWE-918          6  
  CWE-1333         4  
  CWE-502          4  
  CWE-327          4  
  CWE-730          3  
  CWE-400        

## 3 — Validate Structure

In [4]:
REQUIRED_FIELDS = ["lines", "raw_lines", "label", "type", "cwe_id", "dataset_source"]


def validate_split(data: List[Dict], name: str) -> bool:
    issues = []
    for i, r in enumerate(data):
        missing = [f for f in REQUIRED_FIELDS if f not in r]
        if missing:
            issues.append(f"[{i}] missing fields: {missing}")
            continue
        n = len(r["lines"])
        for arr in ("raw_lines", "label", "type"):
            if len(r[arr]) != n:
                issues.append(f"[{i}] {arr} len {len(r[arr])} != lines len {n}")
        bad_lbl = [v for v in r["label"] if v not in (0, 1)]
        if bad_lbl:
            issues.append(f"[{i}] invalid label values: {set(bad_lbl)}")
        cwe = r["cwe_id"]
        if not isinstance(cwe, str) or (
            not cwe.startswith("CWE-") and cwe != "unknown"
        ):
            issues.append(f"[{i}] bad cwe_id: {cwe!r}")
    if issues:
        print(f"FAIL {name}: {len(issues)} issues")
        for issue in issues[:10]:
            print(f"  {issue}")
        return False
    print(f"OK   {name}: {len(data):,} records — all fields valid")
    return True


ok_train = validate_split(train_raw, "FINAL_train")
ok_val = validate_split(val_raw, "FINAL_val")
assert ok_train and ok_val, "Fix validation errors before continuing"

OK   FINAL_train: 3,677 records — all fields valid
OK   FINAL_val: 408 records — all fields valid


## 4 — Normalize & Build UNIFIED Dataset

In [5]:
def normalize(record: Dict, split_origin: str, record_id: int) -> Dict:
    """Add metadata fields to a raw record."""
    r = record.copy()
    r["record_id"] = record_id
    r["split_origin"] = split_origin
    r["num_statements"] = len(r["label"])
    r["num_vulnerable"] = int(sum(r["label"]))
    r["is_vulnerable"] = r["num_vulnerable"] > 0
    return r


norm_train = [normalize(r, "train", i) for i, r in enumerate(train_raw)]
norm_val = [normalize(r, "val", i) for i, r in enumerate(val_raw)]

unified = norm_train + norm_val
for gid, r in enumerate(unified):
    r["global_id"] = gid

# ── Cross-reference mappings ──────────────────────────────────────────────────
mappings: Dict[str, Any] = {
    "by_split": {"train": [], "val": []},
    "by_source": {},
    "by_cwe": {},
    "vulnerable_indices": [],
    "safe_indices": [],
}
for r in unified:
    gid = r["global_id"]
    mappings["by_split"][r["split_origin"]].append(gid)
    mappings["by_source"].setdefault(r["dataset_source"], []).append(gid)
    mappings["by_cwe"].setdefault(r["cwe_id"], []).append(gid)
    (
        mappings["vulnerable_indices"]
        if r["is_vulnerable"]
        else mappings["safe_indices"]
    ).append(gid)

# ── Save files ────────────────────────────────────────────────────────────────
UNIFIED_PATH = DATASETS_DIR / "UNIFIED.jsonl"
MAPPINGS_PATH = DATASETS_DIR / "UNIFIED_mappings.json"
METADATA_PATH = DATASETS_DIR / "UNIFIED_metadata.json"

write_jsonl(UNIFIED_PATH, unified)
print(
    f"Saved UNIFIED.jsonl  — {len(unified):,} records  {UNIFIED_PATH.stat().st_size/1024/1024:.1f} MB"
)

with open(MAPPINGS_PATH, "w") as f:
    json.dump(mappings, f)
print("Saved UNIFIED_mappings.json")

total_stmts_u = sum(r["num_statements"] for r in unified)
vuln_stmts_u = sum(r["num_vulnerable"] for r in unified)
vuln_funcs_u = sum(1 for r in unified if r["is_vulnerable"])
metadata = {
    "total_records": len(unified),
    "total_statements": total_stmts_u,
    "total_vulnerable_statements": vuln_stmts_u,
    "total_vulnerable_functions": vuln_funcs_u,
    "vulnerable_stmt_pct": round(100 * vuln_stmts_u / total_stmts_u, 2),
    "vulnerable_func_pct": round(100 * vuln_funcs_u / len(unified), 2),
    "splits": {k: len(v) for k, v in mappings["by_split"].items()},
    "sources": {k: len(v) for k, v in mappings["by_source"].items()},
    "cwes": {k: len(v) for k, v in mappings["by_cwe"].items()},
    "required_fields": REQUIRED_FIELDS,
    "all_fields": [
        "lines",
        "raw_lines",
        "label",
        "type",
        "cwe_id",
        "dataset_source",
        "record_id",
        "split_origin",
        "global_id",
        "num_statements",
        "num_vulnerable",
        "is_vulnerable",
    ],
}
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved UNIFIED_metadata.json")

Saved UNIFIED.jsonl  — 4,085 records  29.0 MB
Saved UNIFIED_mappings.json
Saved UNIFIED_metadata.json


## 5 — Verify & Print Authoritative Schema

In [7]:
ALL_FIELDS = [
    "lines",
    "raw_lines",
    "label",
    "type",
    "cwe_id",
    "dataset_source",
    "record_id",
    "split_origin",
    "global_id",
    "num_statements",
    "num_vulnerable",
    "is_vulnerable",
]

loaded = read_jsonl(UNIFIED_PATH)
assert len(loaded) == len(
    unified
), f"Record count mismatch: {len(loaded)} vs {len(unified)}"
for i, r in enumerate(loaded):
    missing = [f for f in ALL_FIELDS if f not in r]
    assert not missing, f"Record {i} missing: {missing}"
    assert r["global_id"] == i
    assert r["split_origin"] in ("train", "val")
    assert r["num_statements"] == len(r["lines"])
    assert r["num_vulnerable"] == sum(r["label"])
    assert r["is_vulnerable"] == (r["num_vulnerable"] > 0)

print(f"VERIFIED: {len(loaded):,} records — all 12 fields present and consistent")
print()
print("== UNIFIED RECORD SCHEMA (authoritative) ==")
schema = [
    ("lines", "list[str]", "cleaned code lines"),
    ("raw_lines", "list[str]", "original lines with whitespace/comments preserved"),
    ("label", "list[int]", "0=safe, 1=vulnerable — per statement"),
    ("type", "list[str]", "AST statement type per line"),
    ("cwe_id", "str", 'CWE-NNN  or  "unknown"  (~50% unknown)'),
    ("dataset_source", "str", '"vudenc" | "funclevel" | "securityeval"'),
    ("record_id", "int", "unique ID within split (0-based)"),
    ("split_origin", "str", '"train" | "val"'),
    ("global_id", "int", "unique ID across unified dataset (0-based)"),
    ("num_statements", "int", "len(lines)"),
    ("num_vulnerable", "int", "sum(label)"),
    ("is_vulnerable", "bool", "num_vulnerable > 0"),
]
for field, ftype, desc in schema:
    print(f"  {field:<18} {ftype:<14} {desc}")

unknown_n = sum(1 for r in loaded if r["cwe_id"] == "unknown")
print()
print(
    f'NOTE: {unknown_n:,} / {len(loaded):,} records ({100*unknown_n/len(loaded):.1f}%) have cwe_id="unknown".'
)
print("  These are treated as class 7 in CWE head (0-6 = known CWEs, 7 = unknown).")
print("  All records (known + unknown) contribute to CWE training loss.")

VERIFIED: 4,085 records — all 12 fields present and consistent

== UNIFIED RECORD SCHEMA (authoritative) ==
  lines              list[str]      cleaned code lines
  raw_lines          list[str]      original lines with whitespace/comments preserved
  label              list[int]      0=safe, 1=vulnerable — per statement
  type               list[str]      AST statement type per line
  cwe_id             str            CWE-NNN  or  "unknown"  (~50% unknown)
  dataset_source     str            "vudenc" | "funclevel" | "securityeval"
  record_id          int            unique ID within split (0-based)
  split_origin       str            "train" | "val"
  global_id          int            unique ID across unified dataset (0-based)
  num_statements     int            len(lines)
  num_vulnerable     int            sum(label)
  is_vulnerable      bool           num_vulnerable > 0

NOTE: 2,036 / 4,085 records (49.8%) have cwe_id="unknown".
  These are treated as class 7 in CWE head (0-6 = know

## 6 — CWE Classification Head (8 classes: 7 known + unknown)

In [8]:
import torch
import torch.nn as nn

CWE_7_CLASSES = [
    "CWE-077",
    "CWE-601",
    "CWE-022",
    "CWE-094",
    "CWE-089",
    "CWE-352",
    "CWE-079",
]
CWE_8_CLASSES = CWE_7_CLASSES + ["unknown"]  # 8th class
CWE_TO_INDEX = {cwe: i for i, cwe in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE = {i: cwe for cwe, i in CWE_TO_INDEX.items()}


def map_cwe_to_label(cwe_id: str) -> int:
    """Returns 0-7 for all CWE types: 0-6 known, 7 for unknown."""
    return CWE_TO_INDEX.get(cwe_id, CWE_TO_INDEX["unknown"])


class CWEClassificationHead(nn.Module):
    """8-class CWE head. All records (including unknown) contribute to training."""

    def __init__(self, hidden_size: int, num_classes: int = 8, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, cls_repr: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.dropout(cls_repr))


# Coverage stats
cwe_counts = Counter(r["cwe_id"] for r in loaded)
print("CWE head — training coverage (8 classes):")
for cwe in CWE_8_CLASSES:
    idx = CWE_TO_INDEX[cwe]
    cnt = cwe_counts.get(cwe, 0)
    print(f"  {cwe:<10} label={idx}  n={cnt:,}")
total_labeled = sum(cwe_counts.get(c, 0) for c in CWE_8_CLASSES)
print(
    f"Total training samples: {total_labeled:,} / {len(loaded):,}  ({100*total_labeled/len(loaded):.1f}%)"
)

CWE head — training coverage (8 classes):
  CWE-077    label=0  n=194
  CWE-601    label=1  n=209
  CWE-022    label=2  n=232
  CWE-094    label=3  n=122
  CWE-089    label=4  n=589
  CWE-352    label=5  n=181
  CWE-079    label=6  n=404
  unknown    label=7  n=2,036
Total training samples: 3,967 / 4,085  (97.1%)


## 7 — Model: LoRA-Wrapped GraphCodeBERT

Two implementations are provided — auto-selected by platform:
- **Apple Silicon**: MLX-native LoRA (runs on Metal GPU, ARM-optimized)
- **CUDA / CPU**: PyTorch + PEFT LoRA (standard, Colab-compatible)

In [9]:
# ── Shared utilities ─────────────────────────────────────────────────────────


def count_params(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total, 100 * trainable / total if total else 0.0


# ── [PyTorch] CUDA / CPU / MPS fallback ─────────────────────────────────────
# Used on: Google Colab (CUDA), Linux, Intel Mac, Apple Silicon (fallback)
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model


class GraphCodeBERTLoRACWEModel(nn.Module):
    """
    GraphCodeBERT + LoRA adapters on query+value projections (~0.24% trainable).
    Returns logits (B,8), hidden_states (B,L,H), and optionally loss.
    hidden_states exposed for future binary-detection and line-localization heads.
    """

    def __init__(
        self,
        model_name: str = "microsoft/graphcodebert-base",
        num_cwe_classes: int = 8,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.1,
    ):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["query", "value"],
            bias="none",
        )
        self.encoder = get_peft_model(encoder, lora_cfg)
        self.cwe_head = CWEClassificationHead(
            self.encoder.config.hidden_size, num_cwe_classes
        )
        self.cwe_loss_fn = nn.CrossEntropyLoss()  # all 8 classes, no ignore

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        cwe_labels: torch.Tensor = None,
    ) -> Dict[str, torch.Tensor]:
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = enc_out.last_hidden_state  # (B, L, H)
        cls_repr = hidden_states[:, 0, :]  # (B, H)
        logits = self.cwe_head(cls_repr)  # (B, 8)
        result = {"logits": logits, "hidden_states": hidden_states}
        if cwe_labels is not None:
            result["loss"] = self.cwe_loss_fn(logits, cwe_labels)
        return result


print("PyTorch model class defined: GraphCodeBERTLoRACWEModel")

/Users/anas/Projects/code-security-identifier/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch model class defined: GraphCodeBERTLoRACWEModel


In [10]:
# ── [MLX] Apple Silicon only — skip on other platforms ───────────────────────
# Runs natively on Metal GPU (M1/M2/M3). No CUDA required.
# Uses mlx arrays instead of torch tensors throughout.

if IS_APPLE_SILICON:
    import mlx.core as mx
    import mlx.nn as mlx_nn
    from mlx.utils import tree_flatten, tree_unflatten

    class LinearMLX(mlx_nn.Module):
        """Simple linear layer in MLX."""

        def __init__(self, in_features: int, out_features: int):
            super().__init__()
            self.weight = mx.random.normal((out_features, in_features)) * 0.02
            self.bias = mx.zeros((out_features,))

        def __call__(self, x):
            return x @ self.weight.T + self.bias

    class LoRALinearMLX(mlx_nn.Module):
        """
        LoRA adapter wrapping a frozen linear layer.
        output = W*x  +  (B*A)*x * (alpha/r)
        Only A and B are trainable; W is frozen.
        """

        def __init__(self, frozen_weight, r: int = 8, alpha: int = 16):
            super().__init__()
            out_dim, in_dim = frozen_weight.shape
            self.frozen_W = frozen_weight  # not a parameter — frozen
            self.lora_A = mx.random.normal((r, in_dim)) * 0.02
            self.lora_B = mx.zeros((out_dim, r))
            self.scale = alpha / r

        def __call__(self, x):
            base = x @ self.frozen_W.T
            delta = (x @ self.lora_A.T) @ self.lora_B.T
            return base + delta * self.scale

    class CWEHeadMLX(mlx_nn.Module):
        """8-class CWE classification head in MLX."""

        def __init__(
            self, hidden_size: int, num_classes: int = 8, dropout: float = 0.1
        ):
            super().__init__()
            self.fc = LinearMLX(hidden_size, num_classes)
            self.dropout = dropout

        def __call__(self, x, training: bool = False):
            if training and self.dropout > 0:
                x = mlx_nn.Dropout(p=self.dropout)(x)
            return self.fc(x)

    def cross_entropy_mlx(logits: mx.array, labels: mx.array) -> mx.array:
        """
        Cross-entropy loss in MLX for 8 classes.
        logits: (B, C)  labels: (B,)  — all records contribute
        """
        log_p = mlx_nn.log_softmax(logits, axis=-1)
        # gather log-prob at target class
        B = logits.shape[0]
        idx = mx.arange(B)
        chosen = log_p[idx, labels]  # (B,)
        loss = -(chosen).mean()
        return loss

    class GraphCodeBERTLoRACWEModelMLX(mlx_nn.Module):
        """
        [Apple Silicon — MLX]
        Lightweight stub: random-init encoder-shaped weights + LoRA adapters.
        Intended for smoke-testing and on-device fine-tuning on M-series Macs.

        For production training with the real GraphCodeBERT weights, use the
        PyTorch implementation above on a CUDA GPU (Colab T4/A100).

        hidden_size=768 matches microsoft/graphcodebert-base.
        """

        HIDDEN = 768

        def __init__(
            self, num_cwe_classes: int = 8, lora_r: int = 8, lora_alpha: int = 16
        ):
            super().__init__()
            # Simulated encoder projection (query + value), both LoRA-adapted
            W_q = mx.random.normal((self.HIDDEN, self.HIDDEN)) * 0.02
            W_v = mx.random.normal((self.HIDDEN, self.HIDDEN)) * 0.02
            self.query_lora = LoRALinearMLX(W_q, r=lora_r, alpha=lora_alpha)
            self.value_lora = LoRALinearMLX(W_v, r=lora_r, alpha=lora_alpha)
            self.cwe_head = CWEHeadMLX(self.HIDDEN, num_cwe_classes)

        def __call__(self, hidden_states: mx.array, cwe_labels: mx.array = None):
            """
            hidden_states : (B, L, H) — token representations
            Returns dict with 'logits' (B,8), optionally 'loss' scalar
            """
            cls_repr = hidden_states[:, 0, :]  # (B, H)
            cls_repr = self.query_lora(cls_repr) + self.value_lora(cls_repr)
            logits = self.cwe_head(cls_repr)  # (B, 8)
            result = {"logits": logits}
            if cwe_labels is not None:
                result["loss"] = cross_entropy_mlx(logits, cwe_labels)
            return result

    print("[MLX] GraphCodeBERTLoRACWEModelMLX defined — Apple Silicon native")
else:
    print("[MLX] Skipped — not Apple Silicon")

[MLX] GraphCodeBERTLoRACWEModelMLX defined — Apple Silicon native


## 8 — Smoke Test

Runs the appropriate backend:
- Apple Silicon → MLX model (Metal GPU)
- CUDA available → PyTorch CUDA
- Otherwise → PyTorch CPU

In [11]:
# ── [MLX] Smoke test — Apple Silicon only ────────────────────────────────────
if IS_APPLE_SILICON:
    import mlx.core as mx

    print("[MLX Smoke Test]")
    B, L, H = 2, 16, 768
    fake_hidden = mx.random.normal((B, L, H))
    cwe_labels = mx.array([map_cwe_to_label("CWE-089"), map_cwe_to_label("unknown")])
    assert cwe_labels[0].item() == 4, "CWE-089 → 4"
    assert cwe_labels[1].item() == 7, "unknown → 7"

    mlx_model = GraphCodeBERTLoRACWEModelMLX()
    out = mlx_model(fake_hidden, cwe_labels=cwe_labels)
    mx.eval(out["logits"])  # force Metal execution

    assert out["logits"].shape == (B, 8), f'logits shape wrong: {out["logits"].shape}'
    assert "loss" in out
    loss_val = float(out["loss"])
    assert not (loss_val != loss_val), "NaN loss"  # NaN check

    preds = out["logits"].argmax(axis=-1).tolist()
    print(f'  logits shape : {tuple(out["logits"].shape)}')
    print(f"  loss         : {loss_val:.4f}")
    for i, p in enumerate(preds):
        print(f"  sample {i} pred : {INDEX_TO_CWE[p]}")

    # Trainable param count (MLX: only arrays with requires_grad equivalent)
    params = tree_flatten(mlx_model.trainable_parameters())
    n_trainable = sum(p.size for _, p in params)
    print(f"  MLX trainable params : {n_trainable:,}")
    print("MLX SMOKE TEST PASSED")
else:
    print("[MLX] Skipped — not Apple Silicon")

[MLX Smoke Test]
  logits shape : (2, 8)
  loss         : 1.5924
  sample 0 pred : CWE-089
  sample 1 pred : CWE-089
  MLX trainable params : 1,210,376
MLX SMOKE TEST PASSED


In [12]:
# ── [PyTorch] Smoke test — CUDA / MPS / CPU ──────────────────────────────────
import torch

if torch.cuda.is_available():
    device = "cuda"
elif IS_APPLE_SILICON and torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon MPS fallback (PyTorch path)
else:
    device = "cpu"

print(f"[PyTorch Smoke Test]  device={device}")

model_name = "microsoft/graphcodebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
pt_model = GraphCodeBERTLoRACWEModel(model_name=model_name).to(device)

trainable, total, pct = count_params(pt_model)
print(f"  Trainable: {trainable:,} / {total:,}  ({pct:.2f}%)")

samples = [
    "def get_user(name): return db.execute('SELECT * FROM users WHERE name=' + name)",
    "def add(a, b): return a + b",
]
cwe_labels = torch.tensor(
    [map_cwe_to_label("CWE-089"), map_cwe_to_label("unknown")],
    dtype=torch.long,
    device=device,
)
assert cwe_labels[0].item() == 4, "CWE-089 should map to index 4"
assert cwe_labels[1].item() == 7, "unknown → 7"

enc = tokenizer(
    samples, padding=True, truncation=True, max_length=256, return_tensors="pt"
)
enc = {k: v.to(device) for k, v in enc.items()}

pt_model.eval()
with torch.no_grad():
    out = pt_model(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        cwe_labels=cwe_labels,
    )

assert out["logits"].shape == (2, 8), f'logits shape: {out["logits"].shape}'
assert out["hidden_states"].shape[0] == 2
assert out["hidden_states"].shape[2] == 768
assert "loss" in out
assert not torch.isnan(out["loss"]), "NaN loss"

preds = out["logits"].argmax(dim=-1).tolist()
print(f'  logits shape     : {tuple(out["logits"].shape)}')
print(f'  hidden_states    : {tuple(out["hidden_states"].shape)}')
print(f'  loss             : {out["loss"].item():.4f}')
for i, (s, p) in enumerate(zip(samples, preds)):
    print(f"  sample {i} pred   : {INDEX_TO_CWE[p]}")
print("PyTorch SMOKE TEST PASSED")

[PyTorch Smoke Test]  device=mps


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 71869.00it/s]
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Trainable: 301,064 / 124,946,696  (0.24%)
  logits shape     : (2, 8)
  hidden_states    : (2, 24, 768)
  loss             : 1.8949
  sample 0 pred   : unknown
  sample 1 pred   : CWE-089
PyTorch SMOKE TEST PASSED
